# 04 - WoE / IV (fine classing)

Compute Weight of Evidence and Information Value for every feature, **on train only**.

- **Numeric**: up to 10 quantile bins + a `Missing` bin.
- **Categorical**: one bin per label with ≥ 1% of rows; rarer labels → `Rare`; plus `Missing`.
- **WoE** = ln(%good / %bad): positive → lower risk, negative → higher risk.
- **IV** = Σ (%good − %bad) × WoE.

| IV | Predictive power |
|---|---|
| < 0.02 | useless |
| 0.02 – 0.1 | weak |
| 0.1 – 0.3 | medium |
| 0.3 – 0.5 | strong |
| ≥ 0.5 | suspicious (check leakage) |

This is a first pass (fine classing). Merging bins into monotonic groups (coarse classing) is done in `05_feature_selection`.

## 1. Setup

In [1]:
import pickle
import sys
from pathlib import Path

import pandas as pd

# Add project root to sys.path so `src` can be imported from the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.woe import bin_frame, fit_bins, iv_summary, woe_table

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

## 2. Load train set and variable types

In [2]:
# Outputs of 03_train_test_split and 02_data_cleaning
train = pd.read_pickle(DATA_DIR / "train.pkl")
var_types = pd.read_pickle(DATA_DIR / "var_types.pkl")

numeric_cols = var_types.index[var_types["var_type"] == "numeric"].tolist()
categorical_cols = var_types.index[var_types["var_type"] == "categorical"].tolist()
print(f"Train: {train.shape} | numeric: {len(numeric_cols)} | categorical: {len(categorical_cols)}")

Train: (246008, 123) | numeric: 70 | categorical: 51


## 3. Fit bins and compute WoE

In [3]:
# Learn bin edges / kept categories from train
spec = fit_bins(train, numeric_cols, categorical_cols, n_bins=10, min_share=0.01)

# Map every value to its bin, then build one WoE table per feature
binned = bin_frame(train, spec)
tables = {col: woe_table(binned[col], train["TARGET"]) for col in spec}

# Sanity check: every row must land in exactly one bin of every feature
lost = {col: len(train) - int(t["n"].sum()) for col, t in tables.items()}
assert not any(lost.values()), {c: n for c, n in lost.items() if n}
print("OK: no rows lost in binning")

OK: no rows lost in binning


## 4. IV ranking

In [4]:
iv = iv_summary(tables)

# How many features fall into each predictive-power group
iv["strength"].value_counts()

strength
useless    67
weak       50
strong      2
medium      2
Name: count, dtype: int64

In [5]:
# Top 25 features by IV
iv.head(25).round(4)

,iv,n_bins,strength
EXT_SOURCE_3,0.3296,11,strong
EXT_SOURCE_2,0.3030,11,strong
EXT_SOURCE_1,0.1512,11,medium
YEARS_EMPLOYED,0.1116,11,medium
AMT_GOODS_PRICE,0.0913,11,weak
AGE_YEARS,0.0867,10,weak
OCCUPATION_TYPE,0.0754,13,weak
ORGANIZATION_TYPE,0.0596,18,weak
NAME_INCOME_TYPE,0.0559,5,weak
REGION_RATING_CLIENT_W_CITY,0.0516,3,weak


In [6]:
# Features with (almost) no predictive power -> drop candidates
iv[iv["strength"] == "useless"].round(4)

,iv,n_bins,strength
FLOORSMIN_MEDI,0.0194,8,useless
FLOORSMIN_MODE,0.0188,8,useless
FLOORSMIN_AVG,0.0187,8,useless
AMT_REQ_CREDIT_BUREAU_YEAR,0.0177,7,useless
LANDAREA_MEDI,0.0171,11,useless
...,...,...,...
FLAG_DOCUMENT_19,0.0000,2,useless
FLAG_EMAIL,0.0000,2,useless
FLAG_DOCUMENT_5,0.0000,2,useless
FLAG_DOCUMENT_7,0.0000,2,useless


## 5. Inspect WoE tables

Check that bad rate / WoE move in a sensible direction across bins, and how the `Missing` bin behaves.

In [7]:
# Strong external score: expect bad rate to fall steadily as the score rises
tables["EXT_SOURCE_3"].round(4)

,n,share,n_bad,bad_rate,woe,iv
bin,,,,,,
"(-inf, 0.2276]",19723,0.0802,3928,0.1992,-1.0408,0.1331
"(0.2276, 0.3297]",19757,0.0803,2523,0.1277,-0.5110,0.0260
"(0.3297, 0.4084]",20001,0.0813,1956,0.0978,-0.2105,0.0039
"(0.4084, 0.4758]",19842,0.0807,1576,0.0794,0.0176,0.0000
"(0.4758, 0.5353]",19513,0.0793,1206,0.0618,0.2874,0.0058
"(0.5353, 0.5902]",19506,0.0793,1042,0.0534,0.4420,0.0129
"(0.5902, 0.643]",20444,0.0831,990,0.0484,0.5454,0.0197
"(0.643, 0.6941]",19808,0.0805,812,0.0410,0.7197,0.0310
"(0.6941, 0.749]",18902,0.0768,657,0.0348,0.8910,0.0424


In [8]:
# Age: younger clients are expected to be riskier
tables["AGE_YEARS"].round(4)

,n,share,n_bad,bad_rate,woe,iv
bin,,,,,,
"(-inf, 28.14]",24636,0.1001,2868,0.1164,-0.4056,0.0195
"(28.14, 31.99]",24592,0.1000,2714,0.1104,-0.3454,0.0138
"(31.99, 35.96]",24598,0.1000,2414,0.0981,-0.2144,0.0050
"(35.96, 39.45]",24592,0.1000,2173,0.0884,-0.0987,0.0010
"(39.45, 43.1]",24588,0.0999,1978,0.0804,0.0038,0.0000
"(43.1, 47.13]",24626,0.1001,1902,0.0772,0.0480,0.0002
"(47.13, 51.68]",24583,0.0999,1733,0.0705,0.1466,0.0020
"(51.68, 56.02]",24629,0.1001,1554,0.0631,0.2654,0.0063
"(56.02, 60.71]",24600,0.1000,1327,0.0539,0.4318,0.0156


In [9]:
# Categorical example with rare labels merged into 'Rare'
tables["OCCUPATION_TYPE"].round(4)

,n,share,n_bad,bad_rate,woe,iv
bin,,,,,,
Accountants,7882,0.0320,365,0.0463,0.5915,0.0088
Cleaning staff,3741,0.0152,357,0.0954,-0.1844,0.0006
Cooking staff,4767,0.0194,501,0.1051,-0.2912,0.0019
Core staff,22093,0.0898,1410,0.0638,0.2532,0.0052
Drivers,14944,0.0607,1683,0.1126,-0.3682,0.0096
High skill tech staff,9093,0.0370,571,0.0628,0.2700,0.0024
Laborers,44259,0.1799,4657,0.1052,-0.2918,0.0173
Managers,17096,0.0695,1064,0.0622,0.2799,0.0048
Medicine staff,6871,0.0279,480,0.0699,0.1557,0.0006


In [10]:
# Replace the column name to inspect any other feature
tables["YEARS_EMPLOYED"].round(4)

,n,share,n_bad,bad_rate,woe,iv
bin,,,,,,
"(-inf, 0.91]",20275,0.0824,2240,0.1105,-0.3466,0.0114
"(0.91, 1.69]",20363,0.0828,2353,0.1156,-0.3972,0.0154
"(1.69, 2.51]",19966,0.0812,2194,0.1099,-0.3405,0.0109
"(2.51, 3.42]",20207,0.0821,2082,0.1030,-0.2685,0.0066
"(3.42, 4.51]",20215,0.0822,1910,0.0945,-0.1724,0.0026
"(4.51, 5.91]",20153,0.0819,1696,0.0842,-0.0453,0.0002
"(5.91, 7.64]",20140,0.0819,1528,0.0759,0.0673,0.0004
"(7.64, 10.07]",20234,0.0822,1405,0.0694,0.1628,0.0020
"(10.07, 14.62]",20161,0.0820,1182,0.0586,0.3435,0.0084


## 6. Save artifacts

The binning spec is reused to transform the test set with the same bins.

In [11]:
OUTPUT_DIR.mkdir(exist_ok=True)

with open(OUTPUT_DIR / "woe_spec.pkl", "wb") as f:
    pickle.dump(spec, f)
with open(OUTPUT_DIR / "woe_tables.pkl", "wb") as f:
    pickle.dump(tables, f)
iv.to_csv(OUTPUT_DIR / "iv_summary.csv")

print("Saved woe_spec.pkl, woe_tables.pkl, iv_summary.csv to", OUTPUT_DIR)

Saved woe_spec.pkl, woe_tables.pkl, iv_summary.csv to /home/mdtt/Documents/Mdtt/IT/Projects/credit-risk-scorecard/outputs
